In [5]:
import numpy as np
import pandas as pd
import serial
import threading
import time
from IPython.display import clear_output
import os
from datetime import datetime as dt
from serial.tools import list_ports

SERIAL_TIMOUT = 1000 # ms

if not os.path.exists('./log'):
    os.makedirs('./log')
    
from serial.tools import list_ports
df_ports = pd.DataFrame([c.__dict__ for c in list_ports.comports()])
df_ports

,device,name,description,hwid,vid,pid,serial_number,location,manufacturer,product,interface
0,/dev/cu.BLTH,cu.BLTH,n/a,n/a,NaN,NaN,None,None,None,None,None
1,/dev/cu.MotoricasMacBookPro,cu.MotoricasMacBookPro,n/a,n/a,NaN,NaN,None,None,None,None,None
2,/dev/cu.ESP32-BT-Slave,cu.ESP32-BT-Slave,n/a,n/a,NaN,NaN,None,None,None,None,None
3,/dev/cu.BoseQC35II,cu.BoseQC35II,n/a,n/a,NaN,NaN,None,None,None,None,None
4,/dev/cu.ESP32Bluetooth,cu.ESP32Bluetooth,n/a,n/a,NaN,NaN,None,None,None,None,None
5,/dev/cu.Bluetooth-Incoming-Port,cu.Bluetooth-Incoming-Port,n/a,n/a,NaN,NaN,None,None,None,None,None
6,/dev/cu.usbserial-0001,cu.usbserial-0001,CP2102 USB to UART Bridge Controller - CP2102 ...,USB VID:PID=10C4:EA60 SER=0001 LOCATION=20-2,4292.0,60000.0,0001,20-2,Silicon Labs,CP2102 USB to UART Bridge Controller,CP2102 USB to UART Bridge Controller


In [4]:
import ipywidgets as widgets
from IPython.display import display

# Create the dropdown widget
style = {'description_width': 'initial'}
dropdown = widgets.Dropdown(
    options=df_ports.name.unique(),
    description='Select Device:',
    disabled=False,
    style=style
)

# Create a message field
message = widgets.Label(value="")

ser_port = None
def select(b):
    global ser_port
    ser_port = dropdown.value
    message.value = f"Selected: {dropdown.value}"

# Create the button widget
button = widgets.Button(
    description='Select Device',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Select device'
)

# Link the button to the select function
button.on_click( select)

# Use HBox to align the dropdown and button horizontally
hbox = widgets.HBox([dropdown, button])

# Use VBox to stack the HBox and the message field vertically
vbox = widgets.VBox([hbox, message])

# Display the VBox containing the HBox and message field
display(vbox)


NameError: name 'df_ports' is not defined

In [17]:
ser_port = '/dev/cu.usbserial-1430'

# Background parsing

In [1]:
f'{dt.now(): %c}'

NameError: name 'dt' is not defined

In [ ]:
DEBUG = True
PARSER_ALIVE = True
data_line = np.zeros(5, dtype='uint16')
serial_line = None
ser = None

def serial_parser():
    
    STATUS = 'IDLE'
    global DEBUG
    global PARSER_ALIVE
    global serial_line
    global data_line
    global ser

    ser = serial.Serial(ser_port, baudrate=115200, timeout = SERIAL_TIMOUT / 1000)
    time.sleep(1) # wait till buffer fills with some data
    
    # Flushing serial
    iteration = 0;
    if DEBUG:
        print(f'Flushing serial {ser.inWaiting()}')
        
    while (ser.inWaiting() > 50):
        ser.readline()
        iteration += 1;
        if DEBUG:
            print(f'Flushing {iteration} {ser.inWaiting()}', end = ' '*20 + '\r')
    ser.readline()
    # / Flushing serial

    print('Start Serial Reading')
    
    PACK_START_TS = 0;
    
    NUM_ERRORS = 0;
    MAX_ERRORS = 3;
    
    with open(os.path.join('log', dt.now().strftime("%Y-%m-%d_%H_%M_%S_serial_logging.log")), 'w+') as f:
        f.write(f'System Time: {dt.now().strftime("%Y %m %d %H %M %s")}\r\n')
        log_start = time.time()

        while True:
            if not (PARSER_ALIVE):
                # HALT RUNNING
                STATUS = 'not PARSER_ALIVE'
                ser.close()
                return -1

            STATUS = 'RUNNING'

            try:
                serial_line = ser.readline()[:-2].decode()
                if serial_line is None:
                    PARSER_ALIVE = False
                    STATUS = 'Serial Timeout Error'
                    ser.close()
                    return -1

                data_line[:] = np.array(serial_line.split(' ')).astype('uint16')      
                
                NUM_ERRORS = 0;

            except Exception as e:
                NUM_ERRORS += 1
                if NUM_ERRORS >= MAX_ERRORS:
                    print(e)
                    print(serial_line)
                    PARSER_ALIVE = False
                    STATUS = 'Parsing error'
                    ser.close()
                    return -1
                          
            f.write(f'{time.time() - log_start:.3f} {serial_line} {str(data_line)} {NUM_ERRORS}\r\n')

            if DEBUG:
                print(data_line, end = '                      \r')
        
serial_parser() # testing

# PARSER_ALIVE = False # Kill process

Flushing serial 754
Start Serial Reading               



# Launch in multithread

In [3]:
p = threading.Thread(target=serial_parser, args=())
p.start()
time.sleep(2)  

Flushing serial 0
Start Serial Reading


In [4]:
DEBUG = False

In [41]:
PARSER_ALIVE = False # Kill thread

# Launch main processing

In [5]:
while PARSER_ALIVE:
    print(data_line, end = " "*10 + '\r')
    # Do something time-consuming
    time.sleep(1)

KeyboardInterrupt: 